# Zomato End-to-End Streaming MLOps Pipeline on AWS

This notebook documents and demonstrates the architecture and workflow of the Zomato streaming MLOps pipeline running on AWS EC2. It covers:
- High-level architecture diagram
- Component explanations
- Example code to interact with S3 and MLflow
- Pipeline status and metrics visualization

## 1. High-Level Architecture Diagram

Below is a diagram of the pipeline components and their interactions:

```
+-------------------+        +-------------------+        +-------------------+
|   Kafka Producer  | -----> |   Kafka Broker    | -----> |   Kafka Consumer  |
+-------------------+        +-------------------+        +-------------------+
                                                              |
                                                              v
                                                    +-------------------+
                                                    |      S3 Bucket    |
                                                    +-------------------+
                                                              |
                                                              v
                                                    +-------------------+
                                                    |   Model Training  |
                                                    +-------------------+
                                                              |
                                                              v
                                                    +-------------------+
                                                    |     MLflow UI     |
                                                    +-------------------+
```

- All components (except S3) run on the same EC2 instance.
- S3 is used for data lake and model artifact storage.
- MLflow UI is accessible via browser for experiment tracking.

## 2. Component Explanations

- **Kafka Producer**: Generates synthetic Zomato orders and streams them to a Kafka topic.
- **Kafka Broker**: Handles message streaming between producer and consumer.
- **Kafka Consumer**: Reads messages from Kafka, batches them, and writes to S3 as Parquet files.
- **S3 Bucket**: Stores raw order data and trained model artifacts.
- **Model Training**: Loads data from S3, performs signal audit, trains XGBoost model, logs metrics and artifacts to MLflow.
- **MLflow UI**: Tracks experiments, metrics, and artifacts. Accessible at `http://<EC2-IP>:5000`.

In [ ]:
# 3. Example: Connect to S3 and List Recent Parquet Files
import boto3
import os

s3 = boto3.client('s3', region_name='eu-north-1')
bucket = 'zomatoovertomato'
prefix = 'raw-orders/year=2026/month=02/day=28/'

response = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)
files = [obj['Key'] for obj in response.get('Contents', []) if obj['Key'].endswith('.parquet')]

print('Recent Parquet files in S3:')
for f in files[-5:]:
    print(f)

In [ ]:
# 4. Example: Query MLflow for Recent Runs and Metrics
import mlflow
from mlflow.tracking import MlflowClient

mlflow.set_tracking_uri('http://localhost:5000')  # or your EC2 public IP
client = MlflowClient()

experiment = client.get_experiment_by_name('zomato-kpt-prediction')
runs = client.search_runs(experiment.experiment_id, order_by=["attributes.start_time DESC"], max_results=5)

for run in runs:
    print(f"Run ID: {run.info.run_id}")
    print(f"  Metrics: {run.data.metrics}")
    print(f"  Params: {run.data.params}")
    print(f"  Start Time: {run.info.start_time}")
    print('-'*40)

## 5. Visualize Recent Model Metrics from MLflow

You can use pandas and matplotlib to visualize recent model performance metrics from MLflow runs.

In [ ]:
# Visualize test MAE and RMSE for recent runs
import pandas as pd
import matplotlib.pyplot as plt

# Collect metrics from MLflow runs (reuse code from previous cell)
metrics = []
for run in runs:
    metrics.append({
        'run_id': run.info.run_id,
        'test_mae': run.data.metrics.get('test_mae'),
        'test_rmse': run.data.metrics.get('test_rmse'),
        'start_time': run.info.start_time
    })
df = pd.DataFrame(metrics)

plt.figure(figsize=(8,4))
plt.plot(df['start_time'], df['test_mae'], marker='o', label='Test MAE')
plt.plot(df['start_time'], df['test_rmse'], marker='x', label='Test RMSE')
plt.xlabel('Run Start Time')
plt.ylabel('Metric Value')
plt.title('Recent Model Performance (Test MAE & RMSE)')
plt.legend()
plt.grid(True)
plt.show()